# Transformer POS taggers on the APLS gold data

Compares three transformer approaches with the taggers already in this folder (Stanford left3words, Stanford bidirectional, Flair):

1. **QCRI** `QCRI/bert-base-multilingual-cased-pos-english`: a ready-made BERT POS tagger, run as-is (no training).
2. **`bert-base-cased` + linear layer**: BERT is frozen; only a single `nn.Linear(768, n_tags)` is trained on its outputs.
3. **`roberta-base` + linear layer**: same setup with RoBERTa.

The linear layers are scored with 5-fold cross-validation grouped by turn, so every gold word is tagged by a model that never saw it. There's also a cross-interview check (train on one interview, test on the other).
Scoring uses the same exact-match rule as `get_accuracy_scores.py`.

**Run on Colab with a GPU** (Runtime → Change runtime type → A100 or L4).

In [2]:
!pip install -q transformers

In [3]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU - this will be slow)")

device: cuda NVIDIA A100-SXM4-40GB


## Data location
Put the two `*_with_golden_manual.csv` files in a Google Drive folder and set `DATA_DIR`, or use the upload cell instead.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/POS-tagging"   # <- change to your folder

# Alternative: upload the two CSVs directly (comment out the drive lines above)
# from google.colab import files; files.upload(); DATA_DIR = "."

Mounted at /content/drive


In [4]:
from nltk.tokenize import TreebankWordTokenizer

FILES = {"CB01": "CB01interview1_with_golden_manual.csv",
         "HD20": "HD20interview1_with_golden_manual.csv"}
OLD_TAGGERS = ["pos_tags_left3words", "pos_tags_bidirectional", "flair_pos_tags"]

dfs = {}
for name, fn in FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fn))
    df = df[df["token"].notna()].reset_index(drop=True)            # 6 empty rows in CB01 (never gold)
    df["token"] = df["token"].astype(str).replace({"TRUE": "true"})  # Excel turned "true" into TRUE
    dfs[name] = df
    print(name, len(df), "tokens,", df["golden"].notna().sum(), "gold")

# Split contractions the same way nltk.word_tokenize did for the Stanford run:
# i'm -> i 'm, don't -> do n't, wanna -> wan na. One tag per piece, joined with spaces.
_tb = TreebankWordTokenizer()
def split_word(w):
    return _tb.tokenize(w) or [w]

# One row per piece; one list of pieces per turn (the model sees the whole turn as context).
rows, turn_pieces = [], []
for name, df in dfs.items():
    for turn, g in df.groupby("turn", sort=True):
        pcs = []
        for ridx, tok, gold in zip(g.index, g["token"], g["golden"]):
            sp = split_word(tok)
            gtags = gold.split() if isinstance(gold, str) else None
            for k, p in enumerate(sp):
                # train label only when piece count matches gold tag count
                gp = gtags[k] if gtags is not None and len(gtags) == len(sp) else None
                rows.append((name, turn, ridx, k, p, gp))
            pcs.extend(sp)
        turn_pieces.append(pcs)
P = pd.DataFrame(rows, columns=["file", "turn", "row", "k", "piece", "gold_piece"])
print(len(P), "pieces in", len(turn_pieces), "turns;", P["gold_piece"].notna().sum(), "gold pieces usable for training")

CB01 11271 tokens, 3375 gold
HD20 8855 tokens, 1013 gold
21177 pieces in 1818 turns; 4569 gold pieces usable for training


In [5]:
def to_words(piece_preds, col):
    """Join per-piece predictions back into one tag string per word, stored on dfs[file][col]."""
    tmp = P[["file", "row"]].assign(pred=list(piece_preds))
    joined = tmp.groupby(["file", "row"], sort=False)["pred"].agg(" ".join)
    for name, df in dfs.items():
        df[col] = joined.loc[name].reindex(df.index)

def accuracy(df, col):
    g = df[df["golden"].notna()]
    return 100 * (g[col] == g["golden"]).mean()

def top_confusions(col, n=10):
    g = pd.concat([d[d["golden"].notna()] for d in dfs.values()])
    wrong = g[g[col] != g["golden"]]
    return wrong.groupby(["golden", col]).size().nlargest(n).rename("count").reset_index()

# Fold assignment: whole turns go to one fold, so a turn is never on both sides.
N_FOLDS = 5
turn_keys = P[["file", "turn"]].drop_duplicates().sample(frac=1, random_state=SEED).reset_index(drop=True)
turn_keys["fold"] = turn_keys.index % N_FOLDS
P = P.merge(turn_keys, on=["file", "turn"], how="left")

## Encoder pass
Each turn goes through the model once. Each piece's output is taken from its first subword.

In [6]:
CHUNK = 150   # pieces per forward pass; the longest turn is ~140 pieces, so this almost never splits

@torch.no_grad()
def run_encoder(tokenizer, model, mode):
    """mode='logits' -> predicted label id per piece; mode='emb' -> mean of last 4 hidden layers per piece."""
    model.eval().to(device)
    out = []
    for pcs in turn_pieces:
        for s in range(0, len(pcs), CHUNK):
            chunk = pcs[s:s + CHUNK]
            enc = tokenizer(chunk, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=512)
            wids = enc.word_ids()
            res = model(**{k: v.to(device) for k, v in enc.items()}, output_hidden_states=(mode == "emb"))
            if mode == "logits":
                vals = res.logits[0].argmax(-1).cpu()
            else:
                vals = torch.stack(res.hidden_states[-4:]).mean(0)[0].float().cpu()
            first = {}
            for i, w in enumerate(wids):
                if w is not None and w not in first:
                    first[w] = i
            assert len(first) == len(chunk), "a piece was truncated"
            out.extend(vals[first[j]] for j in range(len(chunk)))
    return out

## 1. QCRI BERT POS tagger (no training)

In [7]:
name = "QCRI/bert-base-multilingual-cased-pos-english"
tok = AutoTokenizer.from_pretrained(name)
mdl = AutoModelForTokenClassification.from_pretrained(name)
ids = run_encoder(tok, mdl, "logits")
to_words([mdl.config.id2label[int(i)] for i in ids], "qcri_bert")
del mdl; torch.cuda.empty_cache()
print({n: round(accuracy(d, "qcri_bert"), 2) for n, d in dfs.items()})

config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  712MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: QCRI/bert-base-multilingual-cased-pos-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  712MB            

model.safetensors: downloading bytes:           |  0.00B            

{'CB01': np.float64(87.08), 'HD20': np.float64(84.7)}


## 2. Frozen BERT / RoBERTa features

In [8]:
FEATS = {}
for key, name, kw in [("bert", "bert-base-cased", {}),
                      ("roberta", "roberta-base", {"add_prefix_space": True})]:
    tok = AutoTokenizer.from_pretrained(name, **kw)
    mdl = AutoModel.from_pretrained(name)
    FEATS[key] = torch.stack(run_encoder(tok, mdl, "emb"))
    del mdl; torch.cuda.empty_cache()
    print(key, tuple(FEATS[key].shape))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

... (9 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta (21177, 768)


## 3. Linear tagging layer
Features are standardized with the training split's mean and std. Then one `nn.Linear` is trained with full-batch AdamW.

In [9]:
TAGS = sorted(P["gold_piece"].dropna().unique())
tag2id = {t: i for i, t in enumerate(TAGS)}
EPOCHS, LR, WD = 300, 1e-2, 1e-2

def train_and_predict(X, train_mask, test_mask):
    """Train a linear layer on gold pieces in train_mask; return tag predictions for pieces in test_mask."""
    tr = train_mask & P["gold_piece"].notna().values
    Xtr = X[tr].to(device)
    ytr = torch.tensor([tag2id[t] for t in P.loc[tr, "gold_piece"]], device=device)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    lin = torch.nn.Linear(X.shape[1], len(TAGS)).to(device)
    opt = torch.optim.AdamW(lin.parameters(), lr=LR, weight_decay=WD)
    for _ in range(EPOCHS):
        opt.zero_grad()
        loss = F.cross_entropy(lin((Xtr - mu) / sd), ytr)
        loss.backward(); opt.step()
    with torch.no_grad():
        pred = lin((X[test_mask].to(device) - mu) / sd).argmax(-1).cpu()
    return [TAGS[i] for i in pred], loss.item()

fold = P["fold"].values
for key, X in FEATS.items():
    preds = np.empty(len(P), dtype=object)
    for f in range(N_FOLDS):
        p, loss = train_and_predict(X, fold != f, fold == f)
        preds[fold == f] = p
        print(f"{key} fold {f}: final train loss {loss:.3f}")
    to_words(preds, f"{key}_linear_cv")

bert fold 0: final train loss 0.001
bert fold 1: final train loss 0.001
bert fold 2: final train loss 0.001
bert fold 3: final train loss 0.001
bert fold 4: final train loss 0.001
roberta fold 0: final train loss 0.001
roberta fold 1: final train loss 0.002
roberta fold 2: final train loss 0.002
roberta fold 3: final train loss 0.001
roberta fold 4: final train loss 0.001


## Results: 5-fold cross-validation (same gold rows as the README)

In [10]:
MODELS = OLD_TAGGERS + ["qcri_bert", "bert_linear_cv", "roberta_linear_cv"]
results = pd.DataFrame({n: [accuracy(d, m) for m in MODELS] for n, d in dfs.items()}, index=MODELS).round(2)
results

,CB01,HD20
pos_tags_left3words,91.56,89.93
pos_tags_bidirectional,91.88,91.31
flair_pos_tags,96.30,94.77
qcri_bert,87.08,84.70
bert_linear_cv,91.73,89.73
roberta_linear_cv,89.33,87.66


## Cross-interview check
Train the linear layer on one interview and test on the other.

In [11]:
file_arr = P["file"].values
rows_x = []
for key, X in FEATS.items():
    for tr_f, te_f in [("CB01", "HD20"), ("HD20", "CB01")]:
        p, _ = train_and_predict(X, file_arr == tr_f, file_arr == te_f)
        sub = P[file_arr == te_f][["row"]].assign(pred=p)
        words = sub.groupby("row", sort=False)["pred"].agg(" ".join)
        d = dfs[te_f].assign(x=words.reindex(dfs[te_f].index))
        rows_x.append({"model": key, "train": tr_f, "test": te_f, "accuracy": round(accuracy(d, "x"), 2)})
pd.DataFrame(rows_x)

,model,train,test,accuracy
0,bert,CB01,HD20,87.86
1,bert,HD20,CB01,84.80
2,roberta,CB01,HD20,86.48
3,roberta,HD20,CB01,81.45


## Most common errors

In [12]:
for col in ["qcri_bert", "bert_linear_cv", "roberta_linear_cv"]:
    print("\n==", col)
    print(top_confusions(col).to_string(index=False))


== qcri_bert
golden qcri_bert  count
    UH        NN    160
   NNP        NN     63
    UH        RB     54
    RB        IN     45
   VBP        VB     41
    UH        IN     31
    IN        TO     21
    UH        JJ     17
    UH        ``     13
   WDT        IN     10

== bert_linear_cv
golden bert_linear_cv  count
    NN            NNP     23
   NNP             NN     22
    JJ             NN     15
    IN             RB     13
    NN             JJ     12
    RB             IN     12
    NN             UH     10
    IN             TO      9
    JJ             RB      9
    RB             JJ      9

== roberta_linear_cv
golden roberta_linear_cv  count
   NNP                NN     21
    JJ                NN     17
    NN                JJ     16
    NN               NNP     16
    NN                UH     15
    RB                JJ     14
    RB                IN     13
    IN                RB     11
    CC                UH     10
    RB                UH     10


In [13]:
for name, df in dfs.items():
    out = os.path.join(DATA_DIR, FILES[name].replace("_with_golden_manual.csv", "_with_transformer_preds.csv"))
    df.to_csv(out, index=False)
    print("saved", out)

saved /content/drive/MyDrive/POS-tagging/CB01interview1_with_transformer_preds.csv
saved /content/drive/MyDrive/POS-tagging/HD20interview1_with_transformer_preds.csv
